# Python → Rust with Claude Opus 5 and GPT-5

Two frontier models are handed the same job: take a Python program and rewrite it as a
single-file Rust program that prints byte-for-byte identical output as fast as this machine
allows. The port is then compiled with `rustc` and executed, so "did it work?" is not a
judgement call — either the binary prints the same numbers, or it does not.

Both models are reached through [OpenRouter](https://openrouter.ai): one client, one key, one
code path, and switching models is a change of string.

| Contender | OpenRouter model ID |
| --- | --- |
| Claude | `anthropic/claude-opus-5` |
| GPT | `openai/gpt-5` |

## Before you run anything

**A Rust toolchain.** `rustc` has to be on your `PATH`; install one from
[rustup.rs](https://rustup.rs) if it isn't, then restart the kernel so the new `PATH` is
picked up. No Cargo project is involved — the generated file is compiled on its own against
the standard library, which is why the prompt forbids external crates.

**An OpenRouter key** in the project's `.env` as `OPENROUTER_API_KEY`. Both contenders are
frontier-priced and `effort="high"` bills the thinking too: one pass over both runs about
25-35 cents, most of it Opus 5's reasoning tokens. The constants cell lists cheap stand-ins,
and `port(..., effort="low")` cuts the bill sharply on either model.

**Patience for the baseline.** The Python program takes about 20 seconds before Rust ever
enters the picture. That slowness is the entire point.

In [ ]:
# Imports

import os
import platform
import subprocess
import time
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from openai.types.chat import (
    ChatCompletionMessageParam as Message,
    ChatCompletionSystemMessageParam as SystemMessage,
    ChatCompletionUserMessageParam as UserMessage,
)

In [ ]:
# One key, one client — OpenRouter speaks the OpenAI protocol, so the OpenAI SDK is the
# client for both models. override=True lets an edited .env win over a stale shell export.
load_dotenv(override=True)

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise RuntimeError(
        "OPENROUTER_API_KEY is not set. Put it in the .env file at the project root "
        "(keys live at https://openrouter.ai/keys) and rerun this cell."
    )
print(f"OpenRouter key loaded, begins {api_key[:8]}")

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)

In [ ]:
# Constants

CLAUDE = "anthropic/claude-opus-5"
GPT = "openai/gpt-5"

# Both contenders are top-tier and priced to match. To get the plumbing working for a
# fraction of a cent, swap in the small siblings — nothing else in the notebook changes:
# CLAUDE, GPT = "anthropic/claude-haiku-4.5", "openai/gpt-5-nano"

# Generous, because on a reasoning model this ceiling covers the thinking as well as the
# answer, and a port that runs out of budget mid-function fails to compile for a silly reason.
# You are billed for tokens produced, not for the ceiling.
MAX_TOKENS = 32_000

## Telling the model where the code will land

A port is only as good as its target. Two things go into every prompt: what this machine is,
and the exact command that will compile the reply. The host target triple and the CPU model
are what decide whether `-C target-cpu=native` unlocks AVX2 or AVX-512; the compile command is
what tells the model there is no Cargo project to add a dependency to.

In [ ]:
def system_info() -> str:
    """Describe this machine and its Rust toolchain, for the porting prompt."""
    try:
        rustc = subprocess.run(
            ["rustc", "-vV"], capture_output=True, text=True, check=True
        ).stdout
    except FileNotFoundError:
        raise RuntimeError(
            "rustc is not on PATH. Install a toolchain from https://rustup.rs, then "
            "restart the kernel — a running kernel keeps the PATH it started with."
        ) from None

    # platform.processor() returns the bare architecture on Linux; the marketing name that
    # actually tells a model which instruction sets exist is in /proc/cpuinfo.
    cpu = platform.processor() or platform.machine()
    cpuinfo = Path("/proc/cpuinfo")
    if cpuinfo.exists():
        for line in cpuinfo.read_text().splitlines():
            if line.startswith("model name"):
                cpu = line.split(":", 1)[1].strip()
                break

    return (
        f"OS: {platform.platform()}\n"
        f"CPU: {cpu} ({os.cpu_count()} logical cores)\n"
        f"Python: {platform.python_implementation()} {platform.python_version()}\n"
        f"{rustc.strip()}"
    )


SYSTEM_INFO = system_info()
print(SYSTEM_INFO)

In [ ]:
# Where the port is written, and how it is built

BUILD_DIR = Path("rust_build")
BUILD_DIR.mkdir(exist_ok=True)
SOURCE = BUILD_DIR / "main.rs"
BINARY = BUILD_DIR / "main"

# Everything the optimiser has, for one crate with no Cargo: opt-level=3 is -O, target-cpu
# lets it emit instructions this exact chip supports, fat LTO with a single codegen unit lets
# it inline across the whole program including std, and panic=abort drops the unwind tables.
COMPILE_COMMAND = [
    "rustc",
    "--edition=2024",
    "-Copt-level=3",
    "-Ctarget-cpu=native",
    "-Clto=fat",
    "-Ccodegen-units=1",
    "-Cpanic=abort",
    str(SOURCE),
    "-o",
    str(BINARY),
]
RUN_COMMAND = [str(BINARY)]

print(" ".join(COMPILE_COMMAND))

## The prompt

Three of the constraints below exist because the target is Rust rather than C++, and each one
maps to a way a plausible-looking port silently returns the wrong answer:

- **Integer width.** Python's integers grow without limit. Rust's do not, and a release build
  wraps on overflow *without complaining* — `overflow-checks` follows `debug-assertions`, and
  both are off here. A port that picks `u32` where the Python needed 64 bits still compiles,
  still runs, still prints a number.
- **Modulo.** Python's `%` floors toward negative infinity; Rust's truncates toward zero.
  `-7 % 3` is `2` in Python and `-1` in Rust. `rem_euclid` is the operator Python actually has.
- **No crates.** `rustc` is invoked directly on one file, so there is nowhere to declare a
  dependency. `use rand::…` is a compile error, not a slow path.

In [ ]:
SYSTEM_PROMPT = """You port Python programs to Rust.

Reply with the contents of a single Rust source file and nothing else: no prose, no
explanation, no Cargo.toml, no shell commands. Comments inside the code are welcome
wherever a choice is not obvious.

The program must print exactly what the Python program prints — same values, same
formatting, same number of lines — in as little wall-clock time as possible."""


def user_prompt_for(python: str) -> str:
    """Wrap `python` in everything the model needs to emit a file that compiles here."""
    return f"""Port this Python program to Rust. Produce the fastest implementation that
still prints identical output.

Constraints:
- Standard library only. The file is compiled on its own by rustc, with no Cargo project
  and no network access, so a `use` of any external crate simply fails to compile.
- Python integers are arbitrary precision; Rust's are not, and this build wraps silently
  on overflow rather than panicking. Pick widths (i64 / u64 / i128) that reproduce Python's
  arithmetic exactly, and reach for `wrapping_*` only where the Python is itself relying
  on a modulus.
- Python's `%` and `//` floor toward negative infinity; Rust's `%` and `/` truncate toward
  zero. Wherever an operand can be negative, use `rem_euclid` / `div_euclid`.
- Reproduce Python's output formatting exactly, floats included: Python's `{{x:.6f}}` is
  Rust's `{{:.6}}`, and a bare `print(x)` of a float is neither of those.

Your reply is written verbatim to `{SOURCE}` and compiled with:
{" ".join(COMPILE_COMMAND)}

The machine it runs on:
{SYSTEM_INFO}

Python program to port:

```python
{python}
```
"""

In [ ]:
def unfence(text: str) -> str:
    """Strip Markdown code fences from a reply.

    Asked point blank for a bare source file, models still fence it about half the time,
    and a stray ```rust line is a syntax error. Dropping every line that opens with a
    fence also copes with the half-written fence you see mid-stream. It would eat a fence
    inside a Rust doc comment as well — no Python program here ports to one.
    """
    lines = [line for line in text.splitlines() if not line.lstrip().startswith("```")]
    return "\n".join(lines).strip()


def port(model: str, python: str, effort: str = "high") -> str:
    """Stream `model`'s Rust port of `python` into SOURCE, and return the source text.

    OpenRouter's unified `reasoning` block is translated into whatever each provider calls
    the same knob, so one dict covers both contenders. The reasoning itself arrives on
    `delta.reasoning` and is dropped here — only `delta.content` is the file.

    SOURCE is written only once a complete reply is in hand. Each way this can fail ends in
    a file that cannot explain itself — an empty one, or one cut off mid-function — so they
    are caught here, where the cause is still visible, instead of surfacing as a syntax
    error from rustc two cells later. Holding the write also means a failed port leaves the
    previous working one intact rather than clobbering it.
    """
    messages: list[Message] = [
        SystemMessage(role="system", content=SYSTEM_PROMPT),
        UserMessage(role="user", content=user_prompt_for(python)),
    ]
    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=MAX_TOKENS,
        extra_body={"reasoning": {"effort": effort}},
        stream=True,
    )

    handle = display(Markdown(f"Thinking — `{model}` …"), display_id=True)
    reply = ""
    finish_reason = None
    last_redraw = 0.0
    for chunk in stream:
        # A mid-stream failure arrives as an `error` field on an otherwise ordinary chunk,
        # so every chunk is checked rather than only the choice-less ones it usually rides
        # on. Skipping it silently is how a provider error becomes an empty file and a
        # baffling rustc error two cells later.
        error = (chunk.model_extra or {}).get("error")
        if error:
            raise RuntimeError(f"OpenRouter reported an error for {model}: {error}")
        # Otherwise a choice-less chunk is a keep-alive or the closing usage record.
        if not chunk.choices:
            continue
        choice = chunk.choices[0]
        reply += choice.delta.content or ""
        finish_reason = choice.finish_reason or finish_reason
        # Repainting on every token makes the browser, not the model, the bottleneck.
        if time.monotonic() - last_redraw > 0.2:
            handle.update(Markdown(f"```rust\n{unfence(reply)}\n```"))
            last_redraw = time.monotonic()

    code = unfence(reply)
    handle.update(Markdown(f"```rust\n{code}\n```"))

    if not code:
        raise RuntimeError(
            f"{model} streamed no code at all (finish_reason={finish_reason!r}). "
            f"{SOURCE} is untouched — rerun the cell."
        )
    if finish_reason == "length":
        raise RuntimeError(
            f"{model} hit the {MAX_TOKENS:,}-token ceiling, so this port stops mid-file. "
            f"{SOURCE} is untouched. Raise MAX_TOKENS, or lower effort so the thinking "
            "takes a smaller share of the budget."
        )

    SOURCE.write_text(code + "\n", encoding="utf-8")
    return code

In [ ]:
def compile_rust() -> None:
    """Compile SOURCE into BINARY, surfacing rustc's diagnostics either way.

    A failed port is a normal outcome worth reading rather than an exception to swallow:
    rustc's errors name the exact line the model got wrong, and warnings are worth a look
    even on success — an `unused_mut` often marks a loop the model meant to vectorise.
    """
    result = subprocess.run(COMPILE_COMMAND, capture_output=True, text=True)
    if result.stderr:
        print(result.stderr, end="")
    if result.returncode != 0:
        raise RuntimeError(f"rustc exited {result.returncode} — see the errors above.")


def compile_and_run(runs: int = 3) -> float:
    """Compile the current port, run it `runs` times, and return the fastest wall clock.

    Three runs rather than one: the first pays for a cold binary and a cold page cache, and
    seeing all three is the cheapest check that the program is actually deterministic. The
    wall clock is a hair above the time the program reports itself — that gap is process
    startup, and on a port this fast it is no longer negligible.
    """
    compile_rust()
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        result = subprocess.run(RUN_COMMAND, check=True, capture_output=True, text=True)
        times.append(time.perf_counter() - start)
        print(result.stdout, end="")
    best = min(times)
    print(f"Fastest of {runs}: {best:.6f}s wall clock")
    return best


def run_python(source: str) -> float:
    """Execute `source` in a namespace of its own and return its wall-clock time.

    The empty globals dict is not a sandbox — Python inserts `__builtins__` into it
    regardless — it only keeps the example programs from colliding with names in here.
    """
    start = time.perf_counter()
    exec(source, {})
    return time.perf_counter() - start

## The example: a slowly convergent series for π

Two hundred million terms of the Leibniz-ish series, in the most naive loop imaginable. There
is no trap here and no cleverness available beyond what the optimiser can find on its own —
it is a pure measurement of what an interpreted loop costs.

The Python cell takes about 20 seconds.

`PI` is only a string, so this is also where your own code goes: replace it, rerun the four
cells below it, and the rest of the notebook follows.

In [ ]:
PI = """
import time


def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations + 1):
        j = i * param1 - param2
        result -= 1 / j
        j = i * param1 + param2
        result += 1 / j
    return result


start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [ ]:
python_pi = run_python(PI)

In [ ]:
port(CLAUDE, PI)

In [ ]:
claude_pi = compile_and_run()

In [ ]:
port(GPT, PI)

In [ ]:
gpt_pi = compile_and_run()

## Results

Fastest of three runs, wall clock, against CPython on the same machine. Read the speedup as an
order of magnitude rather than a benchmark: one prompt, one sample, one laptop, and a
thermally throttled laptop is perfectly capable of inventing a 20% difference between two
identical binaries.

In [ ]:
RESULTS = {"Claude Opus 5": claude_pi, "GPT-5": gpt_pi}

print(f"{'':<15}{'pi (200M terms)':>20}{'versus CPython':>18}")
print(f"{'CPython':<15}{python_pi:>19.2f}s{'—':>18}")
for label, elapsed in RESULTS.items():
    # Formatted first, then padded as a string, so the speedups share one right edge.
    speedup = f"{python_pi / elapsed:.0f}x"
    print(f"{label:<15}{elapsed:>19.3f}s{speedup:>18}")

## What to look for

**Check the number before the clock.** A port that prints a different result has not been
made faster, it has been made wrong, and the time beside it means nothing. That is the failure
mode worth internalising: nothing raises. The model gets no error, `rustc` gets no error, you
get a fast wrong answer — which makes those twelve decimal places the test, not decoration.

**Watch the last digits of π.** Floating-point addition is not associative, so a port that
splits the sum into lanes or threads to go faster accumulates in a different order and can
differ from CPython in the final digit or two. Rust has no `-ffast-math`, so LLVM will not
reassociate `f64` arithmetic behind the model's back — if the digits move, the model chose to
move them. Whether that still counts as "identical output" is a judgement the prompt hands to
the model, and the two contenders do not necessarily make it the same way.

**Read what each one reached for.** This loop can be made much faster than a literal
transcription — unrolled into independent accumulators, or vectorised outright — and every
step away from a transcription is a step further from any guarantee that the output matches.
Worth checking whether the port that won did so honestly.

**Speed is not the only axis.** Two ports that finish within noise of each other can still
differ in how much `unsafe` they used, whether they allocate inside the hot loop, and whether
the code is something you would keep. Skim `rust_build/main.rs` after each run; it is
overwritten by the next port, so copy anything you want to keep.

## Things to try next

- **Paste in your own Python.** Anything self-contained that prints deterministic output will
  do. The constraints already in the prompt are the ones that bite: no crates, and Rust's
  integers overflow silently where Python's grow without limit. To make the machine catch that
  second one, add `-Coverflow-checks=yes` to `COMPILE_COMMAND` and recompile a port you
  distrust — the silent wrap becomes a panic with a line number.
- **Feed the errors back.** When `compile_rust()` raises, send `rustc`'s stderr to the model
  as a follow-up turn and let it repair its own port. Two or three rounds of that is most of
  what an agentic coding loop is.
- **Ask for threads.** `std::thread` and `std::sync` are in the standard library, so nothing
  in the prompt forbids using all your cores — the models simply are not asked to. Add one
  sentence and rerun.
- **Vary the effort.** `port(GPT, PI, effort="low")` costs a fraction as much and takes a
  fraction as long. Whether it ports any worse is a cheap experiment.
- **Change the target.** Drop `-Ctarget-cpu=native` and recompile the same `main.rs`: the
  difference is the part of the speedup that belongs to your CPU rather than to the model.
- **Add a contender.** Any OpenRouter model ID works in `port()` — `openai/gpt-5-codex` and
  `x-ai/grok-4` are both a one-line change.